# Protein-Nanoparticle Interaction Pipeline Notebook

This notebook runs the hardened protein-nanoparticle ML workflow using the refactored modules in this project.

**Scientific honesty note:** the `Interaction` target in this workflow is a heuristic synthetic pseudo-label derived from nanoparticle `surfcharge` and protein `pI`. These metrics do **not** represent experimentally validated HIV-1 Nef / QD-FRET biology or clinical truth.

In [1]:
from __future__ import annotations

import gc
import json
import os
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

from data_loader import TOXICITY_REQUIRED_COLUMNS, detect_sequence_columns, load_ppi_data, load_toxicity_data
from dataset_builder import build_interaction_dataset
from feature_engineering import extract_protein_feature_table
from model_benchmark import benchmark_models, cross_validate_models
from sequence_cleaner import preprocess_sequences
from utils import ENVIRONMENTAL_CONSTANTS, SYNTHETIC_LABEL_DISCLAIMER, ensure_directory, get_environment_versions, set_global_seed, write_json, write_text

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))
ensure_directory(Path.cwd() / ".mplconfig")

RANDOM_SEED = 42
SEQUENCE_POLICY = "replace"
BENCHMARK_SUBSET_SIZE = 200_000
CV_FOLDS = 5
INSPECTION_SAMPLE_SIZE = 1000
OUTPUT_DIR = ensure_directory("artifacts_notebook")

print("Output directory:", OUTPUT_DIR.resolve())
print(SYNTHETIC_LABEL_DISCLAIMER)

Output directory: /home/victor/Downloads/1.projects/vaishnavmulti/artifacts_notebook
Interaction labels in this project are heuristic synthetic pseudo-labels generated from nanoparticle surfcharge and protein pI. They are not experimentally validated biological or clinical ground truth.


In [2]:
def select_model_columns(interaction_df: pd.DataFrame):
    target_column = "Interaction"
    trace_columns = ["protein_id"]
    categorical_features = [
        column
        for column in interaction_df.columns
        if str(interaction_df[column].dtype) in {"category", "object"} and column != target_column
    ]
    numerical_features = [
        column
        for column in interaction_df.columns
        if pd.api.types.is_numeric_dtype(interaction_df[column]) and column not in {target_column, *trace_columns}
    ]
    feature_columns = [*categorical_features, *numerical_features]
    return feature_columns, categorical_features, numerical_features


def deterministic_subset(df: pd.DataFrame, label_column: str, max_rows: int, seed: int) -> pd.DataFrame:
    if len(df) <= max_rows:
        return df.copy()
    indices = train_test_split(
        df.index,
        train_size=max_rows,
        random_state=seed,
        stratify=df[label_column],
    )[0]
    return df.loc[sorted(indices)].copy()


In [3]:
set_global_seed(RANDOM_SEED)
versions = get_environment_versions()
write_json(OUTPUT_DIR / "reports" / "environment_versions.json", versions)
versions

{'python': '3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]',
 'numpy': '2.4.4',
 'pandas': '3.0.2',
 'scikit_learn': '1.8.0',
 'biopython': '1.87'}

## Phase 1-2: Load Datasets and Validate Schemas

In [4]:
health_dir = ensure_directory(OUTPUT_DIR / "health_reports")
report_dir = ensure_directory(OUTPUT_DIR / "reports")
sequence_dir = ensure_directory(OUTPUT_DIR / "sequence_reports")
feature_dir = ensure_directory(OUTPUT_DIR / "feature_reports")
benchmark_dir = ensure_directory(OUTPUT_DIR / "benchmarks")
dataset_dir = ensure_directory(OUTPUT_DIR / "datasets")

toxicity_df, toxicity_meta = load_toxicity_data(output_dir=health_dir)
ppi_df, ppi_meta = load_ppi_data(output_dir=health_dir)
sequence_detection = detect_sequence_columns(ppi_df)
sequence_selection_report = {
    "selected_columns": sequence_detection.columns,
    "reason": sequence_detection.reason,
    "candidate_scores": sequence_detection.candidate_scores,
}
write_json(report_dir / "sequence_column_selection.json", sequence_selection_report)

print("Toxicity shape:", toxicity_df.shape)
print("PPI shape:", ppi_df.shape)
print("Selected sequence columns:", sequence_detection.columns)
print("Required toxicity columns:", TOXICITY_REQUIRED_COLUMNS)

toxicity_df.head()

Toxicity shape: (881, 11)
PPI shape: (73110, 5)
Selected sequence columns: ['protein_sequences_1', 'protein_sequences_2']
Required toxicity columns: ['NPs', 'coresize', 'hydrosize', 'surfcharge', 'surfarea', 'Ec', 'Expotime', 'dosage', 'e', 'NOxygen', 'class']


,NPs,coresize,hydrosize,surfcharge,surfarea,Ec,Expotime,dosage,e,NOxygen,class
0,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.001,1.61,3,nonToxic
1,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.010,1.61,3,nonToxic
2,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.100,1.61,3,nonToxic
3,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,1.000,1.61,3,nonToxic
4,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,5.000,1.61,3,nonToxic


In [5]:
ppi_df.head()

,protein_sequences_1,protein_sequences_2,source_file,ppi_source_label,source_row_index
0,MSVEMDSSSFIQFDVPEYSSTVLSQLNELRLQGKLCDIIVHIQGQP...,MGDTFIRHIALLGFEKRFVPSQHYVYMFLVKWQDLSEKVVYRRFTE...,negative_protein_sequences.csv,negative_protein_sequences,0
1,MPITRMRMRPWLEMQINSNQIPGLIWINKEEMIFQIPWKHAAKHGW...,MTMPVNGAHKDADLWSSHDKMLAQPLKDSDVEVYNIIKKESNRQRV...,negative_protein_sequences.csv,negative_protein_sequences,1
2,MLCVRGARLKRELDATATVLANRQDESEQSRKRLIEQSREFKKNTP...,MRLTLLCCTWREERMGEEGSELPVCASCGQRIYDGQYLQALNADWH...,negative_protein_sequences.csv,negative_protein_sequences,2
3,MDALESLLDEVALEGLDGLCLPALWSRLETRVPPFPLPLEPCTQEF...,MERLQKQPLTSPGSVSPSRDSSVPGSPSSIVAKMDNQVLGYKDLAA...,negative_protein_sequences.csv,negative_protein_sequences,3
4,MALSRGLPRELAEAVAGGRVLVVGAGGIGCELLKNLVLTGFSHIDL...,MVVMNSLRVILQASPGKLLWRKFQIPRFMPARPCSLYTCTYKTRNR...,negative_protein_sequences.csv,negative_protein_sequences,4


## Phase 3: Sequence Cleaning

In [6]:
cleaned_sequences_df, sequence_log_df, sequence_summary = preprocess_sequences(
    ppi_df,
    seq_columns=sequence_detection.columns,
    policy=SEQUENCE_POLICY,
    output_dir=sequence_dir,
)

assert len(cleaned_sequences_df) > 0, "No cleaned protein sequences were retained."
sequence_summary

{'policy': 'replace',
 'replacement_map': {'U': 'C', 'B': 'D', 'Z': 'E', 'X': 'A'},
 'total_sequences_processed': 146220,
 'valid_as_is': 146055,
 'normalized_only': 0,
 'repaired': 165,
 'dropped': 0,
 'retained_total_count': 146220,
 'retained_unique_count': 10344,
 'duplicate_raw_sequences': 135876,
 'duplicate_cleaned_sequences': 135876,
 'invalid_character_frequency': {'U': 165},
 'failed_examples': []}

In [7]:
sequence_log_df.head()

,source_file,ppi_source_label,source_row_index,source_column,raw_sequence,cleaned_sequence,status,reason,invalid_chars,removed_non_letters,was_normalized,was_repaired,is_retained
0,negative_protein_sequences.csv,negative_protein_sequences,0,protein_sequences_1,MSVEMDSSSFIQFDVPEYSSTVLSQLNELRLQGKLCDIIVHIQGQP...,MSVEMDSSSFIQFDVPEYSSTVLSQLNELRLQGKLCDIIVHIQGQP...,valid_as_is,None,,,False,False,True
1,negative_protein_sequences.csv,negative_protein_sequences,0,protein_sequences_2,MGDTFIRHIALLGFEKRFVPSQHYVYMFLVKWQDLSEKVVYRRFTE...,MGDTFIRHIALLGFEKRFVPSQHYVYMFLVKWQDLSEKVVYRRFTE...,valid_as_is,None,,,False,False,True
2,negative_protein_sequences.csv,negative_protein_sequences,1,protein_sequences_1,MPITRMRMRPWLEMQINSNQIPGLIWINKEEMIFQIPWKHAAKHGW...,MPITRMRMRPWLEMQINSNQIPGLIWINKEEMIFQIPWKHAAKHGW...,valid_as_is,None,,,False,False,True
3,negative_protein_sequences.csv,negative_protein_sequences,1,protein_sequences_2,MTMPVNGAHKDADLWSSHDKMLAQPLKDSDVEVYNIIKKESNRQRV...,MTMPVNGAHKDADLWSSHDKMLAQPLKDSDVEVYNIIKKESNRQRV...,valid_as_is,None,,,False,False,True
4,negative_protein_sequences.csv,negative_protein_sequences,2,protein_sequences_1,MLCVRGARLKRELDATATVLANRQDESEQSRKRLIEQSREFKKNTP...,MLCVRGARLKRELDATATVLANRQDESEQSRKRLIEQSREFKKNTP...,valid_as_is,None,,,False,False,True


## Phase 4: Feature Extraction

In [8]:
feature_table_df, feature_log_df, feature_summary = extract_protein_feature_table(
    cleaned_sequences_df["cleaned_sequence"],
    output_dir=feature_dir,
)
assert len(feature_table_df) > 0, "Protein feature extraction produced no successful rows."

merged_feature_df = feature_table_df.merge(
    cleaned_sequences_df[["protein_id", "cleaned_sequence", "example_raw_sequence", "occurrence_count"]],
    on=["protein_id", "cleaned_sequence"],
    how="left",
)
merged_feature_df.to_csv(feature_dir / "protein_feature_table_enriched.csv", index=False)

feature_summary

{'attempted_sequences': 10344,
 'successful_extractions': 10344,
 'failed_extractions': 0}

In [9]:
merged_feature_df[["protein_id", "cleaned_sequence", "pI", "GRAVY", "molecular_weight", "charge_category_pH_7_4", "sequence_length"]].head()

,protein_id,cleaned_sequence,pI,GRAVY,molecular_weight,charge_category_pH_7_4,sequence_length
0,1,MSVEMDSSSFIQFDVPEYSSTVLSQLNELRLQGKLCDIIVHIQGQP...,5.927363,-0.526587,55979.9788,Negative,504
1,2,MGDTFIRHIALLGFEKRFVPSQHYVYMFLVKWQDLSEKVVYRRFTE...,9.123260,-0.688974,44682.2911,Positive,390
2,3,MPITRMRMRPWLEMQINSNQIPGLIWINKEEMIFQIPWKHAAKHGW...,5.223526,-0.673538,36501.8064,Negative,325
3,4,MTMPVNGAHKDADLWSSHDKMLAQPLKDSDVEVYNIIKKESNRQRV...,7.609797,-0.250932,53082.0193,Positive,483
4,5,MLCVRGARLKRELDATATVLANRQDESEQSRKRLIEQSREFKKNTP...,5.755369,-0.787243,164270.6054,Negative,1505


## Phase 5: Dataset Assembly

In [10]:
interaction_df, interaction_summary = build_interaction_dataset(
    toxicity_df=toxicity_df,
    protein_features_df=merged_feature_df,
    environmental_constants=ENVIRONMENTAL_CONSTANTS,
    output_dir=report_dir,
)

assert len(interaction_df) > 0, "Interaction dataset is empty."
assert interaction_df["Interaction"].nunique() >= 2, "Interaction target must contain at least two classes."

interaction_df.to_parquet(dataset_dir / "interaction_dataset.parquet", index=False)
interaction_df.sample(min(INSPECTION_SAMPLE_SIZE, len(interaction_df)), random_state=RANDOM_SEED).to_csv(
    dataset_dir / "interaction_dataset_sample.csv", index=False
)

interaction_summary

{'rows': 9113064,
 'columns': 48,
 'unique_nanoparticle_rows': 394,
 'unique_nanoparticle_names': 5,
 'unique_proteins': 10344,
 'class_counts': {'0': 7893560, '1': 1219504},
 'positive_percentage': 0.13381931697176713,
 'negative_percentage': 0.8661806830282329,
 'null_counts': {'NPs': 0,
  'coresize': 0,
  'hydrosize': 0,
  'surfcharge': 0,
  'surfarea': 0,
  'Ec': 0,
  'Expotime': 0,
  'dosage': 0,
  'e': 0,
  'NOxygen': 0,
  'class': 0,
  'protein_id': 0,
  'sequence_length': 0,
  'pI': 0,
  'GRAVY': 0,
  'molecular_weight': 0,
  'charge_category_pH_7_4': 0,
  'aromaticity': 0,
  'instability_index': 0,
  'flexibility_mean': 0,
  'helix_fraction': 0,
  'turn_fraction': 0,
  'sheet_fraction': 0,
  'aa_frac_A': 0,
  'aa_frac_C': 0,
  'aa_frac_D': 0,
  'aa_frac_E': 0,
  'aa_frac_F': 0,
  'aa_frac_G': 0,
  'aa_frac_H': 0,
  'aa_frac_I': 0,
  'aa_frac_K': 0,
  'aa_frac_L': 0,
  'aa_frac_M': 0,
  'aa_frac_N': 0,
  'aa_frac_P': 0,
  'aa_frac_Q': 0,
  'aa_frac_R': 0,
  'aa_frac_S': 0,
  'a

In [11]:
interaction_df.head()

,NPs,coresize,hydrosize,surfcharge,surfarea,Ec,Expotime,dosage,e,NOxygen,...,aa_frac_S,aa_frac_T,aa_frac_V,aa_frac_W,aa_frac_Y,Medium_pH,Ionic_Strength,Temperature_C,Temperature_K,Interaction
0,Al2O3,39.700001,267.0,36.299999,64.699997,-1.51,24,0.001,1.61,3,...,0.140873,0.035714,0.069444,0.001984,0.031746,7.4,0.15,25.0,298.149994,1
1,Al2O3,39.700001,267.0,36.299999,64.699997,-1.51,24,0.001,1.61,3,...,0.076923,0.053846,0.053846,0.017949,0.035897,7.4,0.15,25.0,298.149994,0
2,Al2O3,39.700001,267.0,36.299999,64.699997,-1.51,24,0.001,1.61,3,...,0.104615,0.055385,0.046154,0.027692,0.021538,7.4,0.15,25.0,298.149994,1
3,Al2O3,39.700001,267.0,36.299999,64.699997,-1.51,24,0.001,1.61,3,...,0.062112,0.051760,0.062112,0.004141,0.039337,7.4,0.15,25.0,298.149994,0
4,Al2O3,39.700001,267.0,36.299999,64.699997,-1.51,24,0.001,1.61,3,...,0.117608,0.046512,0.039203,0.007973,0.013289,7.4,0.15,25.0,298.149994,1


## Phase 6: Benchmarking

In [12]:
feature_columns, categorical_features, numerical_features = select_model_columns(interaction_df)
model_ready_df = interaction_df[feature_columns + ["Interaction", "protein_id"]].copy()
model_ready_df.head(INSPECTION_SAMPLE_SIZE).to_csv(dataset_dir / "model_ready_dataset.csv", index=False)

benchmark_subset_df = deterministic_subset(
    interaction_df,
    label_column="Interaction",
    max_rows=BENCHMARK_SUBSET_SIZE,
    seed=RANDOM_SEED,
)
benchmark_subset_df.to_csv(dataset_dir / "benchmark_subset.csv", index=False)

X_subset = benchmark_subset_df[feature_columns]
y_subset = benchmark_subset_df["Interaction"]
X_train, X_test, y_train, y_test = train_test_split(
    X_subset,
    y_subset,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y_subset,
)
preprocessor_config = {
    "categorical_features": categorical_features,
    "numerical_features": numerical_features,
}

test_metrics_df, _fitted_models = benchmark_models(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    preprocessor=preprocessor_config,
    output_dir=benchmark_dir,
    random_state=RANDOM_SEED,
)
cv_metrics_df = cross_validate_models(
    X=X_subset,
    y=y_subset,
    preprocessor=preprocessor_config,
    cv=CV_FOLDS,
    random_state=RANDOM_SEED,
    output_dir=benchmark_dir,
)
comparison_df = test_metrics_df.merge(cv_metrics_df, on="model_name", how="left", suffixes=("_test", "_cv"))
comparison_df.to_csv(benchmark_dir / "model_comparison.csv", index=False)

assert test_metrics_df["model_name"].nunique() >= 4, "Benchmark did not train at least four models."
comparison_df

,model_name,status_test,score_method,accuracy,precision,recall,f1,roc_auc,pr_auc,failure_reason_test,...,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,RandomForest,success,predict_proba,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None,...,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,4.965068e-17,1.000000,0.000000
1,HistGradientBoosting,success,predict_proba,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None,...,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,4.965068e-17,1.000000,0.000000
2,LinearSVC,success,decision_function,0.998525,0.993659,0.995330,0.994494,0.999982,0.999879,None,...,0.995701,0.000772,0.995105,0.000926,0.995403,0.000589,0.999985,4.240043e-06,0.999902,0.000031
3,LogisticRegression,success,predict_proba,0.998475,0.994025,0.994582,0.994304,0.999975,0.999827,None,...,0.994768,0.000818,0.994545,0.001224,0.994656,0.000923,0.999971,4.834569e-06,0.999801,0.000035


## Optional Full-Data Linear Baseline Check

This notebook records the same feasibility decision used in the main script: the full-data `LinearSVC` baseline is skipped when the full interaction dataset is too large for the configured safety threshold.

In [13]:
from main import _full_linear_baseline_feasibility

feasible, diagnostics = _full_linear_baseline_feasibility(interaction_df)
full_linear_baseline = {
    "status": "would_run" if feasible else "skipped",
    "failure_reason": None if feasible else (
        "Skipped optional full-data LinearSVC baseline because the estimated peak memory exceeded the configured safety threshold."
    ),
    "feasibility_diagnostics": diagnostics,
}
write_json(report_dir / "full_linear_baseline.json", {"full_linear_baseline": full_linear_baseline})
full_linear_baseline

{'status': 'skipped',
 'failure_reason': 'Skipped optional full-data LinearSVC baseline because the estimated peak memory exceeded the configured safety threshold.',
 'feasibility_diagnostics': {'row_count': 9113064,
  'row_count_limit': 1000000,
  'dataset_bytes': 1603899520,
  'available_memory_bytes': 6358888448,
  'estimated_peak_bytes': 4009748800,
  'threshold_bytes': 4451221913}}

## Final Summary

In [14]:
final_summary = {
    "disclaimer": SYNTHETIC_LABEL_DISCLAIMER,
    "toxicity_schema": toxicity_meta["health_report"],
    "ppi_schema": ppi_meta["health_report"],
    "required_toxicity_columns": TOXICITY_REQUIRED_COLUMNS,
    "selected_sequence_columns": sequence_selection_report,
    "sequence_cleaning_summary": sequence_summary,
    "feature_extraction_summary": feature_summary,
    "dataset_assembly_summary": interaction_summary,
    "benchmark_results": test_metrics_df.to_dict(orient="records"),
    "cross_validation_results": cv_metrics_df.to_dict(orient="records"),
    "full_linear_baseline": full_linear_baseline,
    "saved_artifacts": {
        "environment_versions": str(report_dir / "environment_versions.json"),
        "sequence_log": str(sequence_dir / "sequence_processing_log.csv"),
        "sequence_summary": str(sequence_dir / "sequence_preprocessing_summary.json"),
        "protein_feature_table": str(feature_dir / "protein_feature_table.csv"),
        "feature_failures": str(feature_dir / "protein_feature_failures.csv"),
        "interaction_dataset_parquet": str(dataset_dir / "interaction_dataset.parquet"),
        "interaction_dataset_sample_csv": str(dataset_dir / "interaction_dataset_sample.csv"),
        "benchmark_subset_csv": str(dataset_dir / "benchmark_subset.csv"),
        "metrics_csv": str(benchmark_dir / "model_comparison.csv"),
        "test_metrics_csv": str(benchmark_dir / "test_metrics.csv"),
        "cv_metrics_csv": str(benchmark_dir / "cv_metrics.csv"),
    },
}
write_json(report_dir / "final_summary.json", final_summary)

best_rows = test_metrics_df[test_metrics_df["status"] == "success"].sort_values(by="f1", ascending=False)
best_model_name = best_rows.iloc[0]["model_name"] if not best_rows.empty else "None"
report_text = "\n".join([
    "Protein-Nanoparticle Pipeline Notebook Summary",
    "",
    f"Disclaimer: {SYNTHETIC_LABEL_DISCLAIMER}",
    f"Sequence cleaning policy: {SEQUENCE_POLICY}",
    f"Retained unique cleaned sequences: {sequence_summary['retained_unique_count']}",
    f"Successful protein feature rows: {feature_summary['successful_extractions']}",
    f"Interaction dataset rows: {interaction_summary['rows']}",
    f"Benchmark subset rows: {len(benchmark_subset_df)}",
    f"Best benchmark model by test F1: {best_model_name}",
    f"Outputs written under: {OUTPUT_DIR.resolve()}",
])
write_text(report_dir / "final_summary.txt", report_text)

display(test_metrics_df)
display(cv_metrics_df)
display(pd.DataFrame([interaction_summary]))

,model_name,status,score_method,accuracy,precision,recall,f1,roc_auc,pr_auc,failure_reason,confusion_matrix_path,roc_curve_path,pr_curve_path
2,RandomForest,success,predict_proba,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None,artifacts_notebook/benchmarks/RandomForest/con...,artifacts_notebook/benchmarks/RandomForest/roc...,artifacts_notebook/benchmarks/RandomForest/pr_...
3,HistGradientBoosting,success,predict_proba,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None,artifacts_notebook/benchmarks/HistGradientBoos...,artifacts_notebook/benchmarks/HistGradientBoos...,artifacts_notebook/benchmarks/HistGradientBoos...
1,LinearSVC,success,decision_function,0.998525,0.993659,0.995330,0.994494,0.999982,0.999879,None,artifacts_notebook/benchmarks/LinearSVC/confus...,artifacts_notebook/benchmarks/LinearSVC/roc_cu...,artifacts_notebook/benchmarks/LinearSVC/pr_cur...
0,LogisticRegression,success,predict_proba,0.998475,0.994025,0.994582,0.994304,0.999975,0.999827,None,artifacts_notebook/benchmarks/LogisticRegressi...,artifacts_notebook/benchmarks/LogisticRegressi...,artifacts_notebook/benchmarks/LogisticRegressi...


,model_name,status,failure_reason,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
2,RandomForest,success,None,1.00000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,4.965068e-17,1.000000,0.000000
3,HistGradientBoosting,success,None,1.00000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,4.965068e-17,1.000000,0.000000
1,LinearSVC,success,None,0.99877,0.000158,0.995701,0.000772,0.995105,0.000926,0.995403,0.000589,0.999985,4.240043e-06,0.999902,0.000031
0,LogisticRegression,success,None,0.99857,0.000247,0.994768,0.000818,0.994545,0.001224,0.994656,0.000923,0.999971,4.834569e-06,0.999801,0.000035


,rows,columns,unique_nanoparticle_rows,unique_nanoparticle_names,unique_proteins,class_counts,positive_percentage,negative_percentage,null_counts,label_disclaimer,toxicity_rows,protein_feature_rows,expected_cartesian_size,actual_merged_size
0,9113064,48,394,5,10344,"{'0': 7893560, '1': 1219504}",0.133819,0.866181,"{'NPs': 0, 'coresize': 0, 'hydrosize': 0, 'sur...",Interaction labels in this project are heurist...,881,10344,9113064,9113064


In [15]:
del interaction_df, benchmark_subset_df, X_subset, y_subset, X_train, X_test, y_train, y_test
gc.collect()

21998